# CE541E08 — Unit 5 · Day 42 — for Loops: Reservoir Routing, Flood Frequency, Pipe Network and SCS-CN
| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 5 — Introduction to MATLAB |
| **Session** | Day 42 of 45 |
| **Topics** | for loop · reservoir routing · AMS Weibull · Manning's 10 sections · SCS-CN storm |
---
> **Copy each MATLAB code block and run it in MATLAB Online** at [matlab.mathworks.com](https://matlab.mathworks.com).
> Read the explanation and algorithm first. Verify your output matches the expected output.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
session      = "Day 42"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — The MATLAB for Loop

The for loop in MATLAB is used whenever you need to repeat a block of code a fixed number of times. Syntax:

```matlab
for variable = start:stop
    % body
end
```

Or iterating over a vector:

```matlab
for variable = vector_name
    % body — variable takes each value in turn
end
```

Key difference from Python:
- No `in` keyword, no colon after `for`
- Block ends with `end` (not indentation)
- Index is 1-based: `for i = 1:n` gives i = 1, 2, ..., n

---
## Code Block 1 — Reservoir Routing

### What this code does

We simulate level-pool reservoir routing for a 20-day monsoon inflow sequence. Each day we compute the net change in storage and check for spillway overflow.

### Why each step is taken

**`for day = 1:length(inflow)`:**
Iterates from 1 to 20. On each iteration, `day` holds the current day number.

**`inflow(day)`:**
Accesses the day-th element of the inflow vector. In MATLAB this is 1-based — `inflow(1)` is the first day.

**`if storage >= capacity_Mm3`:**
Checks for overflow. If storage exceeds capacity, the excess becomes spillage and storage is reset to capacity. The `spill` variable accumulates over the simulation.

**`repmat('-',1,44)`:**
Creates a 1×44 character array of dashes — used to print a horizontal separator line.

### Algorithm

```
1. capacity=50, storage=8, outflow=0.8 Mm³/day
   inflow = 20-day sequence

2. for day = 1:length(inflow):
   Qin  = inflow(day)
   net  = Qin - outflow
   storage = storage + net
   if storage >= capacity:
     spill = storage - capacity
     storage = capacity
   Print day, Qin, outflow, net, storage, spill

3. After loop: print total spill
```

### Expected output

```
 Day   Inflow  Outflow      Net    Storage
--------------------------------------------
   1     1.20     0.80    +0.40      8.40
   2     1.80     0.80    +1.00      9.40
   ...
  17     2.10     0.80    +1.30     50.00  FULL
  18     1.80     0.80    +1.00     50.00  SPILL 1.00
  ...
```

In [ ]:
# Copy and run this in MATLAB Online
print("""
% CE541E08 - Day 42: for Loop
% Level-pool reservoir routing
capacity_Mm3 = 50;
storage      = 8;
outflow      = 0.8;   % Mm3/day constant release

inflow = [1.2,1.8,3.4,6.7,12.3,18.5,22.1,19.8,15.4,...
          11.2,8.9,6.7,5.4,4.2,3.1,2.8,2.1,1.8,1.4,1.1];

fprintf('%4s %8s %8s %8s %10s\n','Day','Inflow','Outflow','Net','Storage')
fprintf('%s\n',repmat('-',1,44))

total_spill = 0;
for day = 1:length(inflow)
    Qin     = inflow(day);
    net     = Qin - outflow;
    storage = storage + net;
    spill   = 0;
    if storage >= capacity_Mm3
        spill   = storage - capacity_Mm3;
        storage = capacity_Mm3;
        total_spill = total_spill + spill;
        fprintf('%4d %8.2f %8.2f %8.2f %10.2f  SPILL %.2f\n',...
                day,Qin,outflow,net,storage,spill)
    else
        fprintf('%4d %8.2f %8.2f %8.2f %10.2f\n',...
                day,Qin,outflow,net,storage)
    end
end
fprintf('Total spill: %.2f Mm3  Final storage: %.2f Mm3\n',total_spill,storage)
""")

### 🔁 Try this

Change `outflow = 0.8` to `outflow = 2.0` (increased release for irrigation or flood control).

- Does the reservoir still fill?
- What is the total spill now?
- What does this tell a reservoir operations engineer?

---
## Code Block 2 — Weibull Flood Frequency Table

### What this code does

We perform Weibull flood frequency analysis on a 20-year AMS using a for loop — sorting, computing plotting positions, and printing the complete frequency table.

### Why each step is taken

**`sort(AMS,'descend')`:**
Sorts from largest to smallest so that the highest flood gets rank 1 and the lowest exceedance probability.

**`P = i/(n+1)` inside loop:**
Weibull formula for rank i. Each iteration computes the plotting position for one event. `T = 1/P` gives the return period.

**`interp1`:**
After the table is printed, we use `interp1` to estimate design floods for standard return periods. The x-axis is T (non-uniform), y-axis is flow. We interpolate at T=10, 25, 50.

### Algorithm

```
1. AMS = 20-year peak flow array
2. AMS_s = sort(AMS,'descend')
   n = length(AMS)

3. Print table header

4. for i = 1:n:
   P = i/(n+1)
   T = 1/P
   fprintf rank, flow, P, T

5. interp1(T_array, AMS_s, T_target) → design flood
```

### Expected output

```
Rank   Flow(m3/s)  P(exceed)  T(years)
------------------------------------------
   1         5234     0.0385     26.00
   2         4890     0.0769     13.00
   ...
  20         1234     0.9524      1.05

T=10-yr flood: ~4200 m3/s
T=25-yr flood: ~5000 m3/s
```

In [ ]:
# Copy and run this in MATLAB Online
print("""
% for loop: AMS flood frequency analysis
AMS = [1234,2456,1890,3456,2234,4567,3123,1567,2890,5234,...
       3456,2123,4567,2890,1678,3234,4890,2345,3678,1456];
n   = length(AMS);
AMS_sorted = sort(AMS,'descend');

fprintf('%5s %12s %10s %10s\n','Rank','Flow(m3/s)','P(exceed)','T(years)')
fprintf('%s\n',repmat('-',1,42))

T_array = zeros(1,n);
for i = 1:n
    P = i/(n+1);
    T = 1/P;
    T_array(i) = T;
    fprintf('%5d %12.0f %10.4f %10.2f\n', i, AMS_sorted(i), P, T)
end

% Estimate design floods using linear interpolation
for T_target = [10, 25, 50]
    Q_T = interp1(T_array, AMS_sorted, T_target, 'linear');
    fprintf('T=%2d-yr flood: %.0f m3/s\n', T_target, Q_T)
end
""")

### 🔁 Try this

Print only the top 5 events (ranks 1-5) by changing `for i = 1:n` to `for i = 1:5`.

Then find the T=2 year flood using `interp1`. Is it close to the median of the AMS?

---
## Code Block 3 — Manning's for 10 Pipe Sections

### What this code does

We compute velocity, discharge, and design status for each of 10 pipe sections using a for loop over a matrix of pipe data.

### Why each step is taken

**Storing pipe data as a matrix:**
Each row = one pipe section; columns = [Section, Diameter_mm, Manning_n, Slope]. Accessing `sections(i,2)` gives the diameter of pipe i. This is the standard MATLAB pattern for tabular engineering data.

**`if V>=0.6 && V<=3.0, st='OK'; else, st='REVIEW'; end`:**
MATLAB one-liner if-else. The semicolons separate statements on the same line. `&&` is scalar logical AND.

**`pi*(D/2)^2`:**
For a scalar D, no `.^` needed — `^2` is scalar exponentiation.

### Algorithm

```
1. sections = 10×4 matrix
   [Section, D_mm, n, S] per row

2. Print header row

3. for i = 1:size(sections,1):
   Extract D_mm, n, S from row i
   Compute D, R, V, Q
   Classify status
   Print formatted row
```

### Expected output

```
 Sec  D(mm)   V(m/s)     Q(L/s)  Status
   1    200    0.921      28.84       OK
   2    250    1.073      52.61       OK
   ...
  10    250    0.642      31.50       OK
```

In [ ]:
# Copy and run this in MATLAB Online
print("""
% for loop: Manning's computation for 10 pipe sections
sections = [
    1, 200, 0.013, 0.002;
    2, 250, 0.013, 0.003;
    3, 300, 0.013, 0.001;
    4, 200, 0.015, 0.004;
    5, 250, 0.013, 0.002;
    6, 300, 0.013, 0.0015;
    7, 350, 0.013, 0.001;
    8, 400, 0.013, 0.001;
    9, 150, 0.015, 0.005;
   10, 250, 0.013, 0.002;
];

fprintf('%4s %6s %8s %10s %8s\n','Sec','D(mm)','V(m/s)','Q(L/s)','Status')
for i = 1:size(sections,1)
    D_mm=sections(i,2); n=sections(i,3); S=sections(i,4);
    D=D_mm/1000; R=D/4;
    V=(1/n)*R^(2/3)*S^0.5; Q=V*pi*(D/2)^2;
    if V>=0.6 && V<=3.0, st='OK'; else, st='REVIEW'; end
    fprintf('%4d %6d %8.3f %10.2f %8s\n',i,D_mm,V,Q*1000,st)
end
""")

### 🔁 Try this

Count how many sections have status 'REVIEW'.

Add a counter before the loop: `n_review = 0;`

Inside the loop, after the if-else: `if strcmp(st,'REVIEW'), n_review = n_review+1; end`

`strcmp` compares two strings in MATLAB.

---
## Code Block 4 — SCS-CN Hourly Storm Routing

### What this code does

We apply the SCS-CN method hour by hour to a 12-hour storm using a for loop. `cumsum` computes the running total, and the loop applies the conditional Q formula at each hour.

### Why each step is taken

**`P_cum = cumsum(hourly)`:**
MATLAB's `cumsum` works identically to NumPy's. For `[2.1, 4.5, 8.9, ...]` it gives `[2.1, 6.6, 15.5, ...]`.

**`Q_cum = zeros(size(hourly))`:**
Pre-allocate the output array before the loop — faster than growing the array each iteration.

**`Q_hrly = [Q_cum(1), diff(Q_cum)]`:**
`diff` computes `Q_cum(i) - Q_cum(i-1)` for i=2 to 12. Adding `Q_cum(1)` at the front gives the first hour's incremental runoff. This is equivalent to NumPy's `np.diff(Q_cum, prepend=0)`.

### Algorithm

```
1. hourly = 12-hour storm rainfall (mm/hr)
   CN=75, S=84.7mm, Ia=16.9mm

2. P_cum = cumsum(hourly) → running storm total

3. Q_cum = zeros(1,12)
   for i=1:12:
     if P_cum(i) > Ia:
       Q_cum(i) = (P_cum(i)-Ia)^2/(P_cum(i)-Ia+S)

4. Q_hrly = [Q_cum(1), diff(Q_cum)]
   → incremental runoff per hour

5. Print 5-column table: Hr, P_hrly, P_cum, Q_cum, Q_hrly
```

### Expected output

```
CN=75  S=84.7 mm  Ia=16.9 mm
  Hr  P_hrly   P_cum   Q_cum  Q_hrly
   1     2.1     2.1    0.00    0.00
   2     4.5     6.6    0.00    0.00
   3     8.9    15.5    0.00    0.00
   4    15.6    31.1    3.99    3.99
   ...
  12     2.1   158.8   89.12    1.71
```

In [ ]:
# Copy and run this in MATLAB Online
print("""
% for loop: SCS-CN hourly storm routing
hourly = [2.1,4.5,8.9,15.6,22.3,31.4,28.7,18.9,12.3,7.8,4.2,2.1];
CN=75; S=25400/CN-254; Ia=0.2*S;
P_cum = cumsum(hourly);        % running storm total
Q_cum = zeros(size(hourly));   % pre-allocate

for i = 1:length(hourly)
    if P_cum(i) > Ia
        Q_cum(i) = (P_cum(i)-Ia)^2/(P_cum(i)-Ia+S);
    end
end
Q_hrly = [Q_cum(1), diff(Q_cum)];   % incremental runoff

fprintf('CN=%d  S=%.1f mm  Ia=%.1f mm\n',CN,S,Ia)
fprintf('%4s %8s %8s %8s %8s\n','Hr','P_hrly','P_cum','Q_cum','Q_hrly')
for i=1:length(hourly)
    fprintf('%4d %8.1f %8.1f %8.2f %8.2f\n',...
            i,hourly(i),P_cum(i),Q_cum(i),Q_hrly(i))
end
""")

### 🔁 Try this

Change `CN=75` to `CN=90` and re-run.

- At which hour does runoff begin?
- What is the peak hourly runoff? (Find using `max(Q_hrly)`)
- How does total runoff compare to CN=75?

---
## Session Summary — for Loops

| Concept | MATLAB | Python |
|---|---|---|
| Loop over range | `for i = 1:n` | `for i in range(1,n+1):` |
| Loop over vector | `for x = vec_name` | `for x in vec_name:` |
| Loop over matrix rows | `for i = 1:size(A,1)` | `for i, row in enumerate(A):` |
| Access element | `v(i)` — 1-based | `v[i]` — 0-based |
| Separator line | `repmat('-',1,n)` | `'-'*n` |
| Pre-allocate | `zeros(1,n)` | `np.zeros(n)` |
| Cumulative sum | `cumsum(v)` | `np.cumsum(v)` |
| Differences | `diff(v)` | `np.diff(v)` |
| Interpolation | `interp1(x,y,xi)` | `np.interp(xi,x,y)` |

---
## Day 42 Assignment

Using the reservoir routing from Code Block 1, modify the script to:
1. Report the day when the reservoir first reaches 80% capacity (threshold = 40 Mm³)
2. Report whether the reservoir fills completely or the monsoon ends first
3. Count total number of days the reservoir was at 100% capacity (spilling)

In [ ]:
# Copy and run this in MATLAB Online
print("""
% Day42_Assignment.m
clc; clear;
capacity_Mm3 = 50; storage = 8; outflow = 0.8;
inflow = [1.2,1.8,3.4,6.7,12.3,18.5,22.1,19.8,15.4,...
          11.2,8.9,6.7,5.4,4.2,3.1,2.8,2.1,1.8,1.4,1.1];

day_80pct = 0; days_full = 0; total_spill = 0;
for day = 1:length(inflow)
    storage = storage + inflow(day) - outflow;
    if storage >= capacity_Mm3
        total_spill = total_spill + (storage - capacity_Mm3);
        storage = capacity_Mm3;
        days_full = days_full + 1;
    end
    if day_80pct==0 && storage >= 0.8*capacity_Mm3
        day_80pct = day;
        fprintf('80%% capacity reached on Day %d\n', day)
    end
end
if storage >= capacity_Mm3
    fprintf('Reservoir FULL at end\n')
else
    fprintf('Monsoon ended. Final storage: %.2f Mm3\n', storage)
end
fprintf('Days at full capacity: %d\n', days_full)
fprintf('Total spill: %.2f Mm3\n', total_spill)
""")

---
- [ ] Run all MATLAB blocks in MATLAB Online — verify outputs
- [ ] Save scripts as `.m` files
- [ ] Upload: `Unit5_MATLAB/CE541E08_U5_Day42.ipynb`
- [ ] Commit: `Day 42 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*